
Input: `outputs/step3_models.joblib` (from 03).
Baselines: persistence (next week = this week), seasonal median (same month in training weeks), training median.
Output: `outputs/model_comparison.csv`.

In [ ]:
# Working folder: holds HABs_ScrippsPier_v3.xlsx and the outputs/ folder shared by all steps.
# In Colab the folder is on Google Drive, so every step notebook can read what the previous step saved.
import os
WORK_FOLDER = "/content/drive/MyDrive/BIF633_Progress2"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORK_FOLDER, exist_ok=True)
    os.chdir(WORK_FOLDER)
except ImportError:
    pass  # not running in Colab: use the current folder
os.makedirs("outputs", exist_ok=True)
print("Working folder:", os.getcwd())

In [ ]:
# Load everything saved by the previous step
import joblib
globals().update(joblib.load("outputs/step3_models.joblib"))
print("Loaded outputs/step3_models.joblib")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

# The models predict log(1 + Chl-a). Convert back to mg/m3 before scoring.
def to_mg(log_values):
    return np.clip(np.expm1(np.asarray(log_values)), 0, None)  # same as the team's convert_log_prediction_to_original

def rmse(observed, predicted):
    return np.sqrt(mean_squared_error(observed, predicted))

def mae(observed, predicted):
    return mean_absolute_error(observed, predicted)

# Observed Chl-a (mg/m3) for the training and validation weeks
observed_train = to_mg(y_train)
observed_validation = to_mg(y_validation)

# The three core models and the feature set used for "set C" by each model
core_models = ["Linear Regression", "Random Forest", "SVR"]
set_C_name = {
    "Linear Regression": "C_Linear_Selected",   # 13 features kept after the VIF check
    "Random Forest": "C_Full_Temporal",
    "SVR": "C_Full_Temporal",
}
short_name = {
    "A_Environmental": "A",
    "B_Environmental_Seasonal": "B",
    "C_Full_Temporal": "C",
    "C_Linear_Selected": "C",
}
print("Ready")

In [ ]:
# Baseline 1: persistence (next week = this week)
persistence = validation_df["Chloro_current"].values

# Baseline 2: seasonal median (median of the same month in the training weeks)
median_by_month = pd.Series(observed_train).groupby(train_df["month"].values).median()
seasonal_median = validation_df["month"].map(median_by_month).values

# Baseline 3: one constant (median of all training weeks)
train_median = np.full(len(observed_validation), np.median(observed_train))

baseline_rows = []
for name, prediction in [("Persistence", persistence),
                         ("Seasonal median", seasonal_median),
                         ("Train median", train_median)]:
    baseline_rows.append({
        "Model": name,
        "Type": "Baseline",
        "RMSE": rmse(observed_validation, prediction),
        "MAE": mae(observed_validation, prediction),
    })

# Best feature set of each core model (lowest validation RMSE)
model_rows = []
for model_name in core_models:
    best = (training_results_df[training_results_df["Model"] == model_name]
            .sort_values("RMSE_original").iloc[0])
    model_rows.append({
        "Model": f"{model_name} ({short_name[best['Feature_set']]})",
        "Type": "Model",
        "RMSE": best["RMSE_original"],
        "MAE": best["MAE_original"],
    })

comparison = pd.DataFrame(model_rows + baseline_rows).sort_values("RMSE")
display(comparison.round(3))

In [ ]:
comparison.to_csv("outputs/model_comparison.csv", index=False)
print("Saved outputs/model_comparison.csv")